# TookaBERT rate regression

Same Colab recipe as `notebooks/colab/03-tookabert-finetune.ipynb`. The target is the numeric `rate` instead of the three recommendation classes. There is no MLflow cell.

The split is the same 70% train, 15% validation, and 15% test draw, stratified by `recommendation_status`, seed 2026. Stage 1 trains a new one-number head with the encoder frozen. Stage 2 reloads that head and trains the last encoder block. The loss is mean squared error.

Trainer checkpoints are written to Google Drive, not only to the Colab disk. A rerun of the training cell resumes the unfinished stage from the last Drive checkpoint. A stage whose weights are already in `runs/` is not trained again. If the weights were saved and the score was interrupted, the rerun only scores. Validation metrics go to `metrics.json`. Train-split metrics go to `train_metrics.json`. The test score is `test_metrics.json` under `best/`.

Finished files use a `rate` suffix, so they do not replace `runs/PartAI--TookaBERT-Base--head` or `best/PartAI--TookaBERT-Base`.


In [ ]:
!pip -q install polars gdown


In [ ]:
MODEL_NAME = "PartAI/TookaBERT-Base"
MAX_LENGTH = 128
EPOCHS = 1
BATCH_SIZE = 16
GRAD_ACCUM = 2
HEAD_LR = 1e-3
BLOCK_LR = 2e-5
HEAD_LR_WHEN_UNFROZEN = 1e-4
SEED = 2026
SAVE_STEPS = 1000

DRIVE_DIR = "/content/drive/MyDrive/digikala-experiments"
DRIVE_FILE_ID = "1vCkaGbSrXtlKUsdOrguUtuEYobT5vFxr"
DATA_ZIP = "/content/data/digikala-comments-preprocessed.csv.zip"
LOCAL_CSV = "/content/data/digikala-comments-preprocessed.csv"

LABELS = ["recommended", "not_recommended", "no_idea"]
LABEL_CODE = {label: code for code, label in enumerate(LABELS)}
SPLIT_SEED = 2026
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15
TEST_SHARE = 0.15
SPLITS = ["train", "validation", "test"]
SPLIT_SHARE = {"train": TRAIN_SHARE, "validation": VALIDATION_SHARE, "test": TEST_SHARE}
RATE_MIN = 0.0
RATE_MAX = 5.0
STAGES = ["rate-head", "rate-last-block"]


## Drive and the zipped CSV

Mount Drive and download the shared preprocessed comments file. A zip or CSV already on disk is not downloaded again.


In [ ]:
from pathlib import Path
import json
import zipfile

import numpy as np
import polars as pl
import torch
from google.colab import drive
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)
from transformers.trainer_utils import get_last_checkpoint

import gdown

drive.mount("/content/drive")
zip_path = Path(DATA_ZIP)
zip_path.parent.mkdir(parents=True, exist_ok=True)
if not zip_path.is_file():
    gdown.download(id=DRIVE_FILE_ID, output=str(zip_path), quiet=False)
if not zip_path.is_file():
    raise FileNotFoundError(f"Could not download Drive file {DRIVE_FILE_ID} to {zip_path}.")
with zipfile.ZipFile(zip_path) as archive:
    member = "digikala-comments-preprocessed.csv"
    names = [
        name
        for name in archive.namelist()
        if name.endswith(".csv") and not name.startswith("__MACOSX")
    ]
    if member not in names:
        if len(names) != 1:
            raise FileNotFoundError(f"Expected one CSV in the zip, found {names}")
        member = names[0]
    info = archive.getinfo(member)
    local_csv = Path(LOCAL_CSV)
    local_csv.parent.mkdir(parents=True, exist_ok=True)
    if (not local_csv.is_file()) or local_csv.stat().st_size != info.file_size:
        print("extracting", member, info.file_size, "bytes")
        with archive.open(info) as source, local_csv.open("wb") as target:
            while True:
                chunk = source.read(8 * 1024 * 1024)
                if not chunk:
                    break
                target.write(chunk)
drive_root = Path(DRIVE_DIR)
(drive_root / "runs").mkdir(parents=True, exist_ok=True)
(drive_root / "logs").mkdir(parents=True, exist_ok=True)
(drive_root / "best").mkdir(parents=True, exist_ok=True)
(drive_root / "checkpoints").mkdir(parents=True, exist_ok=True)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("zip:", zip_path, "bytes", zip_path.stat().st_size)
print("csv:", local_csv, "bytes", local_csv.stat().st_size)


## Stratified split

Same draw as the recommendation notebooks: within each `recommendation_status`, 70% train, 15% validation, and the rest test, seed 2026. `rate` is kept as the regression target and is not used to draw the split.


In [ ]:
def fold_digits_for_tokenizer(body: pl.Expr, model_name: str) -> pl.Expr:
    """XLM-R was trained on ASCII digits. ParsBERT and TookaBERT keep Persian digits."""
    if "xlm-roberta" not in model_name.lower():
        return body
    folded = body
    for persian_digit, ascii_digit in zip("۰۱۲۳۴۵۶۷۸۹", "0123456789"):
        folded = folded.str.replace_all(persian_digit, ascii_digit, literal=True)
    return folded


reviews = pl.read_csv(
    LOCAL_CSV,
    columns=["body", "recommendation_status", "rate"],
    schema_overrides={
        "body": pl.String,
        "recommendation_status": pl.String,
        "rate": pl.Float64,
    },
)
reviews = reviews.with_columns(
    fold_digits_for_tokenizer(pl.col("body"), MODEL_NAME).alias("body")
)
labeled = reviews.filter(
    pl.col("body").is_not_null()
    & (pl.col("body").str.strip_chars() != "")
    & pl.col("recommendation_status").is_in(LABELS)
    & pl.col("rate").is_not_null()
    & (pl.col("rate") >= RATE_MIN)
    & (pl.col("rate") <= RATE_MAX)
)

parts = []
for label in LABELS:
    group = labeled.filter(pl.col("recommendation_status") == label)
    shuffled = group.sample(fraction=1.0, shuffle=True, seed=SPLIT_SEED)
    count = shuffled.height
    n_train = int(count * TRAIN_SHARE)
    n_validation = int(count * VALIDATION_SHARE)
    n_test = count - n_train - n_validation
    parts.append(
        shuffled.with_columns(
            pl.Series(
                "split",
                (["train"] * n_train)
                + (["validation"] * n_validation)
                + (["test"] * n_test),
            )
        )
    )
labeled = pl.concat(parts)
print(
    labeled.group_by("split", "recommendation_status")
    .len()
    .sort("split", "recommendation_status")
)
print("rate mean", round(float(labeled.get_column("rate").mean()), 4))


## Regression head, then the last encoder block

The head stage freezes the encoder. The last-block stage trains the top encoder block at `2e-5` and the head at `1e-4`. Scores are MAE, RMSE, and R². The kept model is the one with the lower validation MAE.


In [ ]:
class ReviewSet(Dataset):
    def __init__(self, frame: pl.DataFrame, tokenizer, max_length: int):
        self.texts = frame.get_column("body").to_list()
        self.labels = frame.get_column("rate").to_list()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        encoded = self.tokenizer(
            self.texts[index] or "",
            truncation=True,
            max_length=self.max_length,
        )
        encoded["labels"] = float(self.labels[index])
        return encoded


def regression_bundle(actual, predicted):
    actual = np.asarray(actual, dtype=np.float64).reshape(-1)
    predicted = np.asarray(predicted, dtype=np.float64).reshape(-1)
    error = predicted - actual
    mse = float(np.mean(error * error))
    ss_tot = float(np.sum((actual - actual.mean()) ** 2))
    r2 = 0.0 if ss_tot == 0 else 1.0 - float(np.sum(error * error)) / ss_tot
    clipped = np.clip(predicted, RATE_MIN, RATE_MAX)
    clipped_error = clipped - actual
    return {
        "mae": float(np.mean(np.abs(error))),
        "rmse": float(mse ** 0.5),
        "mse": mse,
        "r2": r2,
        "clipped_mae": float(np.mean(np.abs(clipped_error))),
        "clipped_rmse": float(np.mean(clipped_error * clipped_error) ** 0.5),
        "mean_rate": float(actual.mean()),
        "mean_prediction": float(predicted.mean()),
        "rows": int(actual.shape[0]),
    }


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    scores = regression_bundle(labels, predictions)
    return {"mae": scores["mae"], "rmse": scores["rmse"]}


def encoder_prefix(model) -> str:
    for prefix in ("bert", "roberta", "distilbert", "mobilebert"):
        if hasattr(model, prefix):
            return prefix
    raise AttributeError("This checkpoint has no bert/roberta/distilbert/mobilebert encoder.")


def freeze_encoder(model) -> None:
    prefix = encoder_prefix(model)
    for name, param in model.named_parameters():
        param.requires_grad = not name.startswith(prefix + ".")


def unfreeze_last_block(model) -> None:
    freeze_encoder(model)
    prefix = encoder_prefix(model)
    encoder = getattr(model, prefix).encoder
    layers = getattr(encoder, "layer", None)
    if layers is None:
        layers = encoder.layers
    for param in layers[-1].parameters():
        param.requires_grad = True
    for name, param in model.named_parameters():
        if "classifier" in name or name.startswith("score"):
            param.requires_grad = True


def parameter_groups(model, block_lr: float, head_lr: float):
    head, block = [], []
    prefix = encoder_prefix(model)
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith(prefix + "."):
            block.append(param)
        else:
            head.append(param)
    return [
        {"params": block, "lr": block_lr},
        {"params": head, "lr": head_lr},
    ]


class RegressionTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels").float().view(-1)
        outputs = model(**inputs)
        prediction = outputs.logits.view(-1)
        loss = torch.nn.functional.mse_loss(prediction, labels)
        return (loss, outputs) if return_outputs else loss


def slug(model_name: str) -> str:
    return model_name.replace("/", "--")


def run_dir(stage: str) -> Path:
    return Path(DRIVE_DIR) / "runs" / f"{slug(MODEL_NAME)}--{stage}"


def checkpoint_dir(stage: str) -> Path:
    return Path(DRIVE_DIR) / "checkpoints" / f"{slug(MODEL_NAME)}-{stage}"


def best_dir() -> Path:
    return Path(DRIVE_DIR) / "best" / f"{slug(MODEL_NAME)}--rate"


def load_regressor(source):
    return AutoModelForSequenceClassification.from_pretrained(
        source,
        num_labels=1,
        problem_type="regression",
        ignore_mismatched_sizes=True,
    )


## Train

Run this cell again after an interrupt. A finished stage is skipped. An unfinished stage resumes from its last checkpoint on Drive. Weights are copied to `runs/` before scoring, so a crash during the score does not repeat the epoch.


In [ ]:
import inspect

torch.manual_seed(SEED)
train_frame = labeled.filter(pl.col("split") == "train")
valid_frame = labeled.filter(pl.col("split") == "validation")
test_frame = labeled.filter(pl.col("split") == "test")
print("train", train_frame.height, "validation", valid_frame.height, "test", test_frame.height)


def make_args(stage: str, learning_rate: float) -> TrainingArguments:
    shared = dict(
        output_dir=str(checkpoint_dir(stage)),
        learning_rate=learning_rate,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=2,
        load_best_model_at_end=False,
        fp16=torch.cuda.is_available(),
        logging_steps=200,
        report_to="none",
        seed=SEED,
        dataloader_num_workers=2,
    )
    try:
        return TrainingArguments(eval_strategy="epoch", **shared)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **shared)


def score_frame(model, tokenizer, frame: pl.DataFrame) -> dict:
    holder = ReviewSet(frame, tokenizer, MAX_LENGTH)
    collator = DataCollatorWithPadding(tokenizer)
    shared = dict(
        output_dir="/tmp/score-rate",
        per_device_eval_batch_size=BATCH_SIZE,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )
    try:
        args = TrainingArguments(eval_strategy="no", **shared)
    except TypeError:
        args = TrainingArguments(evaluation_strategy="no", **shared)
    token_kwargs = {}
    if "processing_class" in inspect.signature(Trainer.__init__).parameters:
        token_kwargs["processing_class"] = tokenizer
    else:
        token_kwargs["tokenizer"] = tokenizer
    trainer = Trainer(model=model, args=args, data_collator=collator, **token_kwargs)
    prediction = trainer.predict(holder)
    return regression_bundle(prediction.label_ids, prediction.predictions)


def write_metrics(stage: str, split_name: str, metrics: dict, directory: Path) -> None:
    payload = {"model_name": MODEL_NAME, "stage": stage, "split": split_name, "max_length": MAX_LENGTH, **metrics}
    name = "metrics.json" if split_name == "validation" else f"{split_name}_metrics.json"
    (directory / name).write_text(json.dumps(payload, indent=2))
    log_path = Path(DRIVE_DIR) / "logs" / f"{slug(MODEL_NAME)}--rate.jsonl"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a") as handle:
        handle.write(json.dumps(payload) + "\n")
    print(split_name, {key: round(payload[key], 4) for key in ("mae", "rmse", "r2")})


def promote_if_best(stage_dir: Path, metrics: dict) -> None:
    kept = best_dir()
    kept.mkdir(parents=True, exist_ok=True)
    best_metrics_path = kept / "metrics.json"
    previous = None
    if best_metrics_path.is_file():
        previous = json.loads(best_metrics_path.read_text()).get("mae")
    if previous is not None and metrics["mae"] >= previous:
        print("validation MAE", round(metrics["mae"], 4), "does not beat", round(previous, 4))
        return
    for source in stage_dir.iterdir():
        if source.is_file():
            (kept / source.name).write_bytes(source.read_bytes())
    print("best model updated", kept, "validation MAE", round(metrics["mae"], 4))


def stage_scored(stage: str) -> bool:
    folder = run_dir(stage)
    return (folder / "model.safetensors").is_file() and (folder / "metrics.json").is_file() and (folder / "train_metrics.json").is_file()


def attach_tokenizer(trainer, tokenizer):
    if "processing_class" in inspect.signature(Trainer.__init__).parameters:
        return {"processing_class": tokenizer}
    return {"tokenizer": tokenizer}


for stage in STAGES:
    folder = run_dir(stage)
    if stage_scored(stage):
        print("skip", stage, folder)
        continue
    folder.mkdir(parents=True, exist_ok=True)
    if stage == "rate-head":
        source = MODEL_NAME
    else:
        source = run_dir("rate-head")
        if not (source / "model.safetensors").is_file():
            raise FileNotFoundError(f"Train the head stage first. Missing {source}")
    if (folder / "model.safetensors").is_file():
        print("weights already saved; scoring", stage)
        model = load_regressor(folder)
        tokenizer = AutoTokenizer.from_pretrained(folder)
    else:
        model = load_regressor(source)
        tokenizer = AutoTokenizer.from_pretrained(source if stage == "rate-head" else source)
        if stage == "rate-head":
            freeze_encoder(model)
            learning_rate = HEAD_LR
            groups = None
        else:
            unfreeze_last_block(model)
            learning_rate = BLOCK_LR
            groups = parameter_groups(model, BLOCK_LR, HEAD_LR_WHEN_UNFROZEN)
        trainable = sum(param.numel() for param in model.parameters() if param.requires_grad)
        print(stage, "trainable parameters", trainable)
        train_set = ReviewSet(train_frame, tokenizer, MAX_LENGTH)
        valid_set = ReviewSet(valid_frame, tokenizer, MAX_LENGTH)
        collator = DataCollatorWithPadding(tokenizer)
        trainer = RegressionTrainer(
            model=model,
            args=make_args(stage, learning_rate),
            train_dataset=train_set,
            eval_dataset=valid_set,
            data_collator=collator,
            compute_metrics=compute_metrics,
            **attach_tokenizer(None, tokenizer),
        )
        if groups is not None:
            def create_optimizer(trainer=trainer, groups=groups):
                trainer.optimizer = torch.optim.AdamW(groups, weight_decay=0.01)
                return trainer.optimizer
            trainer.create_optimizer = create_optimizer
        last = get_last_checkpoint(str(checkpoint_dir(stage)))
        print("resume checkpoint", last)
        trainer.train(resume_from_checkpoint=last)
        model.save_pretrained(folder)
        tokenizer.save_pretrained(folder)
        print("saved weights before scoring", folder)
    validation_metrics = score_frame(model, tokenizer, valid_frame)
    write_metrics(stage, "validation", validation_metrics, folder)
    train_metrics = score_frame(model, tokenizer, train_frame)
    write_metrics(stage, "train", train_metrics, folder)
    promote_if_best(folder, validation_metrics)

kept = best_dir()
if not (kept / "model.safetensors").is_file():
    raise FileNotFoundError(f"No best model at {kept}")
if (kept / "test_metrics.json").is_file():
    print("skip test; already saved", kept / "test_metrics.json")
else:
    best_model = load_regressor(kept)
    best_tokenizer = AutoTokenizer.from_pretrained(kept)
    test_metrics = score_frame(best_model, best_tokenizer, test_frame)
    write_metrics("best", "test", test_metrics, kept)
    print("test", {key: round(test_metrics[key], 4) for key in ("mae", "rmse", "r2")})
